### Libraries

In [2]:
# Standard library
import pickle

# Data & Math
import numpy as np
import pandas as pd

# sklearn: Processing & Pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA 
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# sklearn: Model Selection
from sklearn.model_selection import train_test_split


# Local libraries
from utils.skl_lib.LibPreTabTransformer import MaskedPCA, shift_plus_one

# Data split train and test

## Preforming a join 

In [1]:
df_iden = pd.read_csv('../../data_ieee/train_identity.csv')
df_tran = pd.read_csv('../../data_ieee/train_transaction.csv')

df = df_tran.merge(df_iden, on='TransactionID', how='left', suffixes=(None, '_new'))

df.head()

NameError: name 'pd' is not defined

In [3]:
y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pd.concat([X_train, y_train], axis=1).to_csv('../../data_ieee/transactions_train.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../../data_ieee/transactions_test.csv', index=False)


# Making the preprocesing object

### Column spliting

In [4]:
# Split numerical/categorical columns
df = pd.read_csv('../../data_ieee/transactions_train.csv')
df = df.drop(columns=['isFraud'])

cat_cols = df.select_dtypes(include=['object', 'category', 'integer']).columns.tolist()
num_cols = df.select_dtypes(include=['float']).columns.tolist()


print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

# PCA columns
cols_V = [c for c in df_tran.columns if any(x in c[0:2] for x in ['V'])]
cols_C = sorted([c for c in df_tran.columns if any(x in c[0:2] for x in ['C'])])

df_V = df[cols_V]
df_C = df[cols_C]

print("\n\n")
print(f"C_V: {cols_V}")
print(f"C_C: {cols_C}")


Categorical columns: ['TransactionDT', 'ProductCD', 'card1', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Numerical columns: ['TransactionAmt', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', '


Finding columns with similar NaN patterns so PCA is performed within those subgroups.

In [5]:
# Makes a dictionary of columns with the same amount of Nan
nan_series = df_V.isna().sum()
unique_counts = nan_series.unique()
set_V = {}
for count in sorted(unique_counts):
    cols = sorted(nan_series[nan_series == count].index.tolist())
    print(f"Count V{count}: {cols[:5]}...") 
    set_V[f'Count V{count}'] = cols
set_V[f'Count C{count}'] = cols_C
print(f"Count C{count}: {cols_C[:5]}...") 

Count V8: ['V279', 'V280', 'V284', 'V285', 'V286']...
Count V247: ['V100', 'V101', 'V102', 'V103', 'V104']...
Count V1034: ['V281', 'V282', 'V283', 'V288', 'V289']...
Count V60884: ['V12', 'V13', 'V14', 'V15', 'V16']...
Count V61789: ['V53', 'V54', 'V55', 'V56', 'V57']...
Count V71409: ['V75', 'V76', 'V77', 'V78', 'V79']...
Count V135336: ['V35', 'V36', 'V37', 'V38', 'V39']...
Count V223670: ['V1', 'V10', 'V11', 'V2', 'V3']...
Count V359006: ['V220', 'V221', 'V222', 'V227', 'V234']...
Count V360266: ['V169', 'V170', 'V171', 'V174', 'V175']...
Count V360415: ['V167', 'V168', 'V172', 'V173', 'V176']...
Count V367798: ['V217', 'V218', 'V219', 'V223', 'V224']...
Count V406437: ['V322', 'V323', 'V324', 'V325', 'V326']...
Count V406756: ['V143', 'V144', 'V145', 'V150', 'V151']...
Count V406761: ['V138', 'V139', 'V140', 'V141', 'V142']...
Count C406761: ['C1', 'C10', 'C11', 'C12', 'C13']...


## Making the pipeline

In [6]:
# Given the set of groups of PCA columns we make a columns transformer to
# preprocess using the custom PCA class
pca_parallel_steps = []
i = 1
for group_name, cols in set_V.items():
    pca_parallel_steps.append((f'pca_group_{i}', MaskedPCA(n_components=0.9), cols))
    i += 1

pca_engine = ColumnTransformer(pca_parallel_steps, remainder='passthrough')

num_pipeline = Pipeline([
    ("pca", pca_engine),
    ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)), 
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ("shifter", FunctionTransformer(shift_plus_one)) # Ensures that (x >= 0) for pytorch
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

preprocesing_pipeline = preprocessor.fit(X_train)
X_processed = preprocesing_pipeline.transform(X_train)


### Finding the dimensions
Here we use the pipeline to find the dimensions of the categorical and numerical parts.


In [7]:

column_breakdown = {}
current_idx = 0


if "num" in preprocessor.named_transformers_:

    sample_num = preprocessor.named_transformers_['num'].transform(X[num_cols].iloc[:5])
    n_num = sample_num.shape[1]
    column_breakdown['num'] = (current_idx, current_idx + n_num)
    current_idx += n_num


if "cat" in preprocessor.named_transformers_:
    sample_cat = preprocessor.named_transformers_['cat'].transform(X[cat_cols].iloc[:5])
    n_cat = sample_cat.shape[1]
    column_breakdown['cat'] = (current_idx, current_idx + n_cat)
    current_idx += n_cat

print(f"Final Feature Map: {column_breakdown}")

Final Feature Map: {'num': (0, 170), 'cat': (170, 203)}


### Saves the pipeline

In [8]:
preprocesing = {
    'preprocessor': preprocesing_pipeline,
    'num': column_breakdown['num'],
    'cat': column_breakdown['cat']
}


with open('../../models/preprocessing_config.pkl', 'wb') as f:
    pickle.dump(preprocesing, f)

with open('../../models/preprocessing_config.pkl', 'rb') as f:
    loaded_config = pickle.load(f)

pipeline = loaded_config['preprocessor']
numerical_features = loaded_config['num']
categorical_features = loaded_config['cat']

